# 07b — Validate repaired provider queue

Independent deterministic gate. No provider calls are permitted from this notebook.

In [1]:
from pathlib import Path
import sys
import pandas as pd
REPO_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO_ROOT))
from src.geo.m7_query_repair import validate_rebuilt_queue, atomic_parquet, atomic_json
RUN_ROOT = REPO_ROOT / 'data/30_geo/mbn/m7/run_20260808_m7_geo'
queue = pd.read_parquet(RUN_ROOT / 'provider_query_queue_repaired.parquet')
validation = validate_rebuilt_queue(REPO_ROOT, queue)
atomic_parquet(validation, RUN_ROOT / 'repaired_queue_independent_validation.parquet')
summary = {k:int(v) for k,v in {'unsupportedRegion':validation.unsupportedRegion.sum(), 'unsupportedName':validation.unsupportedName.sum(), 'priorityMismatch':(~validation.priorityMatch).sum(), 'brokenArticleFK':validation.brokenArticleFK.sum(), 'brokenBlockFK':validation.brokenBlockFK.sum(), 'brokenEntityFK':validation.brokenEntityFK.sum()}.items()}
summary['verdict'] = 'PRE_PROVIDER_READY_REPAIRED' if validation.gatePass.all() else 'QUERY_EVIDENCE_REPAIR_FAILED'
atomic_json(summary, RUN_ROOT / 'repaired_queue_independent_gate.json')
print(summary)

{'unsupportedRegion': 0, 'unsupportedName': 0, 'priorityMismatch': 0, 'brokenArticleFK': 0, 'brokenBlockFK': 0, 'brokenEntityFK': 0, 'verdict': 'PRE_PROVIDER_READY_REPAIRED'}
